# 🌿 FLOREA V6 — Nature Art Studio

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ArwaKassebi/florea-nature-art-studio/blob/main/notebooks/FLOREA_V6.ipynb)

**Une photo, un son, quelques mots → une œuvre, un poème, une histoire.**

Ce notebook ne contient que l'**exécution sur GPU** : tout le code vit dans [`src/florea/`](../src/florea) (testé par `pytest` sans GPU). Voir le [README](../README.md) pour l'architecture, les résultats et les limites.

⚠️ Avant de commencer : *Exécution → Modifier le type d'exécution → GPU (T4)*, puis exécutez les cellules **dans l'ordre**. Premier lancement : ~5 à 8 min (téléchargement des modèles).

In [ ]:
# 1) Cache des modèles hors de Google Drive + vérification du GPU
import os
os.environ["HF_HOME"] = "/content/hf_cache"
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# 2) Récupérer le dépôt et installer les dépendances (remplacez ArwaKassebi)
REPO = "https://github.com/ArwaKassebi/florea-nature-art-studio.git"
!git clone -q {REPO} /content/florea-nature-art-studio 2>/dev/null || git -C /content/florea-nature-art-studio pull -q
%cd /content/florea-nature-art-studio
!pip -q install -r requirements.txt
!pip -q install -e . --no-deps

In [ ]:
# 3) Tests unitaires (2 à 3 s, sans GPU) : une erreur ici = un vrai bug
!pip -q install pytest
!python -m pytest -q

In [ ]:
# 4) Initialisation : embeddings, FAISS, graphe, GCN (+ évaluation sur liens masqués), modèles enregistrés (chargés à la demande)
from florea import pipeline as P

P.init(
    llm_name="Qwen/Qwen2.5-1.5B-Instruct",   # "…0.5B-Instruct" = rapide | "…3B-Instruct" = meilleure écriture (licence différente)
    save_dir="/content/florea_gallery",      # mettez un dossier Google Drive pour garder vos créations
    use_agent=True,                          # True : l'agent ReAct décide d'appeler la recherche de connaissances
)

## ✅ Test rapide (texte seul, ≈ 30 s)

In [ ]:
r = P.run_florea(user_text="Un champ de lavande au coucher du soleil, des oiseaux chantent, je pense à mon enfance",
                 style_name="✨ Auto (FLOREA choisit)", language="Français", seed=42)
print("\nStyle :", r["style_name"], "| Émotion :", r["emotion"], "| Saison :", r["season"])
print("Classement GCN :", r["ranked_styles"])
print("Trace de l'agent :", r["trace"])
print("Concept :", r["concept"]["title"], "|", r["concept_info"])
print("Prompt image :", r["final_prompt"])
print("\n" + r["poem"])
r["images"][0]
# Audio seul : P.run_florea(audio_path="/content/mon_son.mp3")  |  Photo seule : P.run_florea(image=Image.open("/content/photo.jpg"))

## 🔌 Serveur MCP
Les mêmes outils exposés au standard MCP : un vrai client lance le serveur et l'interroge.

In [ ]:
import sys
sys.path.insert(0, "/content/florea-nature-art-studio/mcp_server")
import demo_client
demo_client.run("/content/florea-nature-art-studio/mcp_server/florea_server.py")

## 🚀 Interface
Cliquez sur le **lien public** (gradio.live). Gardez la cellule en marche.

In [ ]:
from florea import ui
ui.launch(share=True)

---
**Astuces** — *Photo seule* : réglez « Transformation » (0.4 = fidèle, 0.8 = très artistique). *Style Auto* : le GCN classe les styles, FLOREA tire au sort parmi les 3 meilleurs. *Pourquoi cette œuvre ?* : agent, graphe, GCN, prompt exact et temps par étape. *Erreur de mémoire* : message clair, modèles libérés ; réessayez avec moins de variantes ou le LLM 0.5B.